# TypedDict를 이용한 LangGraph 상태 정의 및 그래프 구성과 실행

## 환경 설정

In [3]:
# 필요한 함수 / 클래스 임포트
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph import add_messages, StateGraph, END

## LangGraph 상태 정의

In [4]:
# TypedDict 클래스 상속 -> 클래스 생성 -> 연습

## 클래스 생성
class User(TypedDict):
    id: int
    name: str
    email: str
    is_active: bool

## 딕셔너리 자료형 생성 연습

### 방식1: {} 기호 이용
dict_data1 = {}
dict_data1['name']='젠슨황'
dict_data1['age']=63
print(dict_data1)
print(type(dict_data1))

print('-'*80)

### 방식2: dict() 생성 함수 이용
dict_data2 = dict(name='젠슨황',age=63 )
print(dict_data2)
print(type(dict_data2))

print('-'*80)

### 방식3: 클래스 이용
dict_data3: User = {
    "id": 1,
    "name": "김철수",
    "email": "chulsoo@example.com",
    "is_active": True
}
print(dict_data3)
print(type(dict_data3))

print('-'*80)

### 방식4: 클래스 이용(dict 함수 처럼 사용)
dict_data4 = User(id=2, name="이영희", email="yh@example.com", is_active=False)
print(dict_data4)
print(type(dict_data4))

{'name': '젠슨황', 'age': 63}
<class 'dict'>
--------------------------------------------------------------------------------
{'name': '젠슨황', 'age': 63}
<class 'dict'>
--------------------------------------------------------------------------------
{'id': 1, 'name': '김철수', 'email': 'chulsoo@example.com', 'is_active': True}
<class 'dict'>
--------------------------------------------------------------------------------
{'id': 2, 'name': '이영희', 'email': 'yh@example.com', 'is_active': False}
<class 'dict'>


In [5]:
# LangGraph 상태 정의: TypedDict 클래스 상속

"""
# 첫 번째 항목(key): messages -> 대화 메세지 입력
  1) 대화 메세지: HumanMessage, AIMessage, ToolMessage 
  2-1) 대화 흐름(예시): 사용자 질문 -> LLM 답변(도구 호출 요청) -> 도구 사용
  2-2) 대화 메세지 생성 순서: HumanMessage -> AIMessage -> ToolMessage
  2-3) messages 항목의 값 업데이트
       : [HumanMessage] -> [HumanMessage, AIMessage] -> [HumanMessage, AIMessage, ToolMessage]  
  3) 잘못된 값의 자료형 -> messages: list[BaseMessage] 
                      -> 결과: 기존 대화 메시지가 새로운 대화 메시지로 대체
                      -> 예시: [HumanMessage(content="안녕")] -> [AIMessage(content="반갑습니다! 무엇을 도와드릴까요?")]
# Annotated 클래스 기능
  1) 기존 타입 힌트에 부가 정보(함수, 설명 등)를 추가하는 도구
  2) 코드: Annotated[T(기존 자료형: list[BaseMessage]), 함수, 설명글]

# add_messages 함수 기능
  1) 기존 메시지 리스트와 새 메시지 리스트를 연결 
  2) [HumanMessage] + [AIMessage] -> [HumanMessage, AIMessage] -> 대화 기록 누적 

# 정리: 
  1) 목표: 모든 대화 기록을 messages(키)에 누적해서 저장
  2) 목표 구현 예시: [HumanMessage] -> [HumanMessage, AIMessage] -> [HumanMessage, AIMessage, ToolMessage] 
  3) 구현 방법: messages: Annotated[list[BaseMessage], add_messages]
"""

class AgentState(TypedDict):
    # 대화 메세지 누적 구조 생성
    messages: Annotated[list[BaseMessage], add_messages]
    # 도구 실행 결과 텍스트 저장
    tool_output: str | None

## 노드 함수 정의

In [6]:
# 내용 정리

"""
# 노드를 구성하는 함수
  1) 매개 변수의 자료형: 'AgentState'로 지정
  2) 매개 변수의 입력값 -> {'messages':[BaseMessage], 'tool_output']}
  3) 실행 결과 -> 딕셔너리로 반환
  4) LangGraph 시스템: 실행 결과를 '상태'에 반영 (업데이트) 한다.

# 진행 순서(가정)
  : 사용자 요청 -> call_llm(도구 사용 요청) 함수 -> call_tool(도구 실행) 함수 -> summarize_result(결과 요약) 함수
"""

# llm 호출 함수 정의(Type Hint 적용)
def call_llm(state: AgentState) -> dict:
    print("---LLM 호출 노드 실행---")
    print('-'*80)

    ## state = {'messages':[BaseMessage], 'tool_output']} -> 대화 히스토리 -> 'messages'키의 값 추출
    messages = state["messages"]

    ## messages 상태 (1턴) -> [HumanMessage]
    ## messages 상태 (2턴) -> [HumanMessage, AIMessage, ToolMessage, HumanMessage]

    ## if 조건문 이용, 마지막 메시지가 HumanMessage인지 확인 후 content 값 가져오기
    if messages and isinstance(messages[-1], HumanMessage):
        last_user_message = messages[-1].content
    else:
        last_user_message = ""

    print(f'사용자 입력: {last_user_message}')

    print('-'*80)

    ## 실제: LLM -> '사용자 입력' 분석 -> 도구 사용 할지 말지 판단 -> 가정: 도구 사용을 지시

    ## call_llm 함수 실행의 결과 -> 도구 사용을 지시 -> AIMessage 생성
    ## 실제는 llm 응답 결과를 반환, 여기서는 AIMessage 클래스를 사용해서 직접 결과를 생성
    return {"messages":[AIMessage(content="도구 사용을 요청합니다.")]}

# 도구 실행 함수 정의
def call_tool(state: AgentState) -> dict:
    print("---도구 노드 실행---")

    print('-'*80)

    ## 도구 실행: 여기서는 도구 실행을 시뮬레이션한다.
    print("도구 실행 중...")
    tool_result = "도구 실행 완료."

    ## 함수 실행의 결과 -> 도구 응답(ToolMessage)
    ## 도구 실행의 결과는 ToolMessage 자료형으로 생성, 
    ## 여기서는 ToolMessage 클래스를 이용하여 messages 처리
    return {
        "messages": [ToolMessage(content=tool_result, tool_call_id="searcg_tool_call_id_123")],
        "tool_output": tool_result
    }

# 결과 요약 함수 정의(llm 최종 답변 생성)
def summarize_result(state: AgentState) -> dict:
    print("---결과 요약 노드 실행---")

    print('-'*80)

    ## 상태 -> 도구 실행의 결과 추출
    tool_result = state["tool_output"]

    ## 결과 요약: 실제로는 llm이 수행
    summary = f"도구 실행 결과 '{tool_result}'를 요약했습니다."
    # print(f"\n결과 요약: {summary}")

    ## 함수 실행의 결과 -> LLM의 응답(AIMessage)
    ## 실제는 llm의 요약 결과를 반환, 여기서는 AIMessage 클래스를 사용해서 직접 결과를 생성하고 반환
    return {"messages": [AIMessage(content=summary)]}    

## 그래프(작업 흐름에 대한 설계도) 구성 

In [7]:
# 사용자 요청 -> call_llm -> call_tool -> summarize_result -> 종료

# 그래프 생성 도구 호출, 그래프 생성
graph = StateGraph(AgentState)

# 그래프에 노드 추가 -> StateGraph 클래스 전용 함수 -> add_node()
graph.add_node("call_llm_node", call_llm)
graph.add_node("call_tool_node", call_tool)
graph.add_node("summarize_node", summarize_result)

# 시작점 설정 -> StateGraph 클래스 전용 함수 -> set_entry_point()
graph.set_entry_point("call_llm_node")

# 그래프에 엣지 추가 -> StateGraph 클래스 전용 함수 -> add_edge()

## LLM 실행 후 도구 호출
graph.add_edge("call_llm_node", "call_tool_node")

## 도구 사용 후 결과 요약
graph.add_edge("call_tool_node", "summarize_node")

## 결과 요약 후 종료
graph.add_edge("summarize_node", END)

## 그래프 컴파일

In [8]:
# 컴파일: LangGraph 고유 문법 -> StateGraph 전용 함수 -> compile() 함수 -> 그래프를 실행 가능한 구조(시스템)로 변환 
agent = graph.compile()

## agent 실행 및 결과 확인

In [9]:
print("=== TypedDict 기반 에이전트 실행 ===")

# 사용자 질문 생성
user_question = "nvidia 2026년도 2분기 매출액은?"

# 사용자 질문 -> '상태'에 입력 가능한 형태(규칙 준수) -> dict {"message": [HumanMessage(content=user_question)]} 
initial_state = {"messages": [HumanMessage(content=user_question)]}

# agent 실행 -> 최종 결과 생성
# 결과물 추적: HumanMessage(사용자 요청) -> AIMessage(도구 호출 요청) -> ToolMessage(도구 작동) -> AIMessage(요약 생성) 
# 최종 상태: "messages": [HumanMessage, AIMessage(도구 호출 요청), ToolMessage(도구 작동), AIMessage(요약 생성)] 
final_state = agent.invoke(initial_state)

# 결과 확인
print(f"\n최종 상태: \n{final_state}")
print('-'*80)
print(f"최종 상태 메시지: {final_state['messages']}")
print('-'*80)
print(f"최종 LLM 출력: {final_state['messages'][-1].content}")

=== TypedDict 기반 에이전트 실행 ===
---LLM 호출 노드 실행---
--------------------------------------------------------------------------------
사용자 입력: nvidia 2026년도 2분기 매출액은?
--------------------------------------------------------------------------------
---도구 노드 실행---
--------------------------------------------------------------------------------
도구 실행 중...
---결과 요약 노드 실행---
--------------------------------------------------------------------------------

최종 상태: 
{'messages': [HumanMessage(content='nvidia 2026년도 2분기 매출액은?', additional_kwargs={}, response_metadata={}, id='81ccb142-793c-4171-a8bc-348f33b59688'), AIMessage(content='도구 사용을 요청합니다.', additional_kwargs={}, response_metadata={}, id='1bf996c8-918d-4121-ba3c-d2e7fa4d3017', tool_calls=[], invalid_tool_calls=[]), ToolMessage(content='도구 실행 완료.', id='5026ca86-892d-490a-94a5-273491582c68', tool_call_id='searcg_tool_call_id_123'), AIMessage(content="도구 실행 결과 '도구 실행 완료.'를 요약했습니다.", additional_kwargs={}, response_metadata={}, id='f3c30854-d7aa-4b